In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
!wget -q -O "YT-BB-Dog.zip" "https://www.lirmm.fr/YT-BB-Dog_Sibetan/files/YT-BB-dog.zip"
!ls -lh YT-BB-Dog.zip

-rw-r--r-- 1 root root 583M Oct 28  2025 YT-BB-Dog.zip


In [3]:
!unzip -q "YT-BB-Dog.zip" -d "dataset"
!find dataset -maxdepth 3 | head -20

dataset
dataset/YT-BB-dog
dataset/YT-BB-dog/YT-BB-Dog_random_bckg.zip
dataset/YT-BB-dog/YT-BB-Dog.zip


In [4]:
!unzip -q "dataset/YT-BB-dog/YT-BB-Dog.zip" -d "dataset/perros"
!find dataset/perros -maxdepth 2 | head -10

dataset/perros
dataset/perros/YT-BB-Dog
dataset/perros/YT-BB-Dog/test
dataset/perros/YT-BB-Dog/train


In [5]:
import os
from glob import glob

base = "dataset/perros/YT-BB-Dog"

for parte in ["train", "test"]:
    ruta = os.path.join(base, parte)
    perros = [d for d in os.listdir(ruta) if os.path.isdir(os.path.join(ruta, d))]
    imagenes = glob(os.path.join(ruta, "*", "*.jpg"))
    print(f"{parte.upper()}:")
    print(f"  Perros: {len(perros)}")
    print(f"  Imágenes: {len(imagenes)}")

TRAIN:
  Perros: 2000
  Imágenes: 19932
TEST:
  Perros: 723
  Imágenes: 7104


In [6]:
import os
import random
from glob import glob
import pandas as pd

random.seed(42)

base = "dataset/perros/YT-BB-Dog"

def obtener_perros(ruta):
    perros = []
    for nombre in os.listdir(ruta):
        carpeta = os.path.join(ruta, nombre)
        if os.path.isdir(carpeta):
            imagenes = glob(os.path.join(carpeta, "*.jpg"))
            if len(imagenes) >= 2:      # necesitamos al menos 2 fotos por perro
                perros.append((nombre, imagenes))
    return perros

def crear_pares(perros, pares_por_perro):
    pares = []

    # Mismo perro (etiqueta 1)
    for nombre, imagenes in perros:
        for _ in range(pares_por_perro):
            img1, img2 = random.sample(imagenes, 2)
            pares.append({"imagen1": img1, "imagen2": img2, "label": 1})

    # Perros distintos (etiqueta 0), tantos como los anteriores
    cantidad = len(pares)
    for _ in range(cantidad):
        perro1, imagenes1 = random.choice(perros)
        perro2, imagenes2 = random.choice(perros)
        while perro1 == perro2:
            perro2, imagenes2 = random.choice(perros)
        pares.append({
            "imagen1": random.choice(imagenes1),
            "imagen2": random.choice(imagenes2),
            "label": 0
        })

    df = pd.DataFrame(pares)
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

# Elegimos perros: 400 para entrenar y 100 para probar
todos_train = obtener_perros(os.path.join(base, "train"))
todos_test = obtener_perros(os.path.join(base, "test"))

perros_train = random.sample(todos_train, 400)
perros_test = random.sample(todos_test, 100)

df_train = crear_pares(perros_train, pares_por_perro=10)
df_test = crear_pares(perros_test, pares_por_perro=10)

os.makedirs("pares", exist_ok=True)
df_train.to_csv("pares/pares_train.csv", index=False)
df_test.to_csv("pares/pares_test.csv", index=False)

print("ENTRENAMIENTO:", len(df_train), "pares")
print("  Mismo perro:", int((df_train["label"] == 1).sum()))
print("  Distintos:  ", int((df_train["label"] == 0).sum()))
print("PRUEBA:", len(df_test), "pares")
print("  Mismo perro:", int((df_test["label"] == 1).sum()))
print("  Distintos:  ", int((df_test["label"] == 0).sum()))

ENTRENAMIENTO: 8000 pares
  Mismo perro: 4000
  Distintos:   4000
PRUEBA: 2000 pares
  Mismo perro: 1000
  Distintos:   1000


In [7]:
import cv2
import numpy as np
import pandas as pd

TAMANO = 128

def cargar_imagen(ruta):
    imagen = cv2.imread(ruta)
    if imagen is None:
        raise ValueError(f"No se pudo cargar: {ruta}")
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    imagen = cv2.resize(imagen, (TAMANO, TAMANO))
    return imagen.astype("float32") / 255.0

def preparar_datos(df):
    X1, X2, y = [], [], []
    for _, fila in df.iterrows():
        X1.append(cargar_imagen(fila["imagen1"]))
        X2.append(cargar_imagen(fila["imagen2"]))
        y.append(fila["label"])
    return np.array(X1), np.array(X2), np.array(y, dtype="float32")

df_train = pd.read_csv("pares/pares_train.csv")
df_test = pd.read_csv("pares/pares_test.csv")

X1_train, X2_train, y_train = preparar_datos(df_train)
X1_test, X2_test, y_test = preparar_datos(df_test)

print("ENTRENAMIENTO:", X1_train.shape, X2_train.shape, y_train.shape)
print("PRUEBA:       ", X1_test.shape, X2_test.shape, y_test.shape)

ENTRENAMIENTO: (8000, 128, 128, 3) (8000, 128, 128, 3) (8000,)
PRUEBA:        (2000, 128, 128, 3) (2000, 128, 128, 3) (2000,)


In [8]:
import tensorflow as tf
from tensorflow.keras import layers, Model

@tf.keras.utils.register_keras_serializable(package="Perros")
class AbsDifference(layers.Layer):
    def call(self, inputs):
        return tf.abs(inputs[0] - inputs[1])

def bloque(x, filtros):
    x = layers.Conv2D(filtros, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    return x

# --- Parte que "mira" una foto y saca la huella ---
entrada = layers.Input(shape=(128, 128, 3))

x = layers.RandomFlip("horizontal")(entrada)
x = layers.RandomRotation(0.05)(x)
x = layers.RandomZoom(0.1)(x)

for filtros in [32, 64, 128, 256]:
    x = bloque(x, filtros)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)

extractor = Model(entrada, x, name="extractor_128")

# --- Red siamesa completa ---
imagen_1 = layers.Input(shape=(128, 128, 3), name="imagen_1")
imagen_2 = layers.Input(shape=(128, 128, 3), name="imagen_2")

diferencia = AbsDifference(name="diferencia")(
    [extractor(imagen_1), extractor(imagen_2)]
)
x = layers.Dense(64, activation="relu", name="comparacion")(diferencia)
salida = layers.Dense(1, activation="sigmoid", name="salida")(x)

modelo = Model([imagen_1, imagen_2], salida)

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

modelo.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ imagen_1            │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ imagen_2            │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ extractor_128       │ (None, 128)       │    422,752 │ imagen_1[0][0],   │
│ (Functional)        │                   │            │ imagen_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ diferencia          │ (None, 128)       │          0 │ extractor_128[0]… │
│ (AbsDifference)     │                   │            │ extractor_128[1]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ comparacion (Dense) │ (None, 64)        │      8,256 │ diferencia[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ salida (Dense)      │ (None, 1)         │         65 │ comparacion[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 431,073 (1.64 MB)

 Trainable params: 430,113 (1.64 MB)

 Non-trainable params: 960 (3.75 KB)

In [9]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6
    ),
]

historia = modelo.fit(
    [X1_train, X2_train],
    y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=32,
    callbacks=callbacks,
)

Epoch 1/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 30s 97ms/step - accuracy: 0.7878 - loss: 0.4602 - val_accuracy: 0.6444 - val_loss: 0.5745 - learning_rate: 0.0010
Epoch 2/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.8342 - loss: 0.3793 - val_accuracy: 0.6212 - val_loss: 0.6497 - learning_rate: 0.0010
Epoch 3/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 18s 92ms/step - accuracy: 0.8516 - loss: 0.3361 - val_accuracy: 0.6612 - val_loss: 0.5826 - learning_rate: 0.0010
Epoch 4/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 20s 91ms/step - accuracy: 0.8573 - loss: 0.3370 - val_accuracy: 0.7200 - val_loss: 0.5306 - learning_rate: 0.0010
Epoch 5/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 19s 93ms/step - accuracy: 0.8677 - loss: 0.3203 - val_accuracy: 0.6981 - val_loss: 0.5598 - learning_rate: 0.0010
Epoch 6/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 19s 93ms/step - accuracy: 0.8662 - loss: 0.3127 - val_accuracy: 0.6800 - val_loss: 0.6010 - learning_rate: 0.0010
Epoch 7/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 18s 92ms/step - accuracy: 0.8731 - l

In [10]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

predicciones = modelo.predict([X1_test, X2_test]).ravel()
predicciones_clase = (predicciones >= 0.5).astype(int)

print("RESULTADOS EN PERROS NUEVOS (2000 pares)")
print("Accuracy :", round(accuracy_score(y_test, predicciones_clase), 4))
print("Precision:", round(precision_score(y_test, predicciones_clase), 4))
print("Recall   :", round(recall_score(y_test, predicciones_clase), 4))
print("F1-score :", round(f1_score(y_test, predicciones_clase), 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, predicciones_clase))

63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step
RESULTADOS EN PERROS NUEVOS (2000 pares)
Accuracy : 0.7185
Precision: 0.9823
Recall   : 0.445
F1-score : 0.6125

Matriz de confusión:
[[992   8]
 [555 445]]


In [11]:
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

# Los últimos 1600 pares de train fueron la validación (20 %)
X1_val, X2_val, y_val = X1_train[6400:], X2_train[6400:], y_train[6400:]

pred_val = modelo.predict([X1_val, X2_val]).ravel()

mejor_umbral, mejor_f1 = 0.5, 0
for umbral in np.arange(0.05, 0.96, 0.05):
    f1 = f1_score(y_val, (pred_val >= umbral).astype(int))
    print(f"umbral {umbral:.2f} -> F1 {f1:.4f}")
    if f1 > mejor_f1:
        mejor_umbral, mejor_f1 = umbral, f1

print(f"\nMejor umbral: {mejor_umbral:.2f}")

# Probar ese umbral en los perros nuevos (test)
clase = (predicciones >= mejor_umbral).astype(int)

print("\nRESULTADOS EN TEST CON EL NUEVO UMBRAL")
print("Accuracy :", round(accuracy_score(y_test, clase), 4))
print("Precision:", round(precision_score(y_test, clase), 4))
print("Recall   :", round(recall_score(y_test, clase), 4))
print("F1-score :", round(f1_score(y_test, clase), 4))
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, clase))

50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step
umbral 0.05 -> F1 0.8599
umbral 0.10 -> F1 0.8926
umbral 0.15 -> F1 0.9014
umbral 0.20 -> F1 0.8981
umbral 0.25 -> F1 0.8813
umbral 0.30 -> F1 0.8647
umbral 0.35 -> F1 0.8290
umbral 0.40 -> F1 0.7967
umbral 0.45 -> F1 0.7449
umbral 0.50 -> F1 0.6805
umbral 0.55 -> F1 0.6000
umbral 0.60 -> F1 0.5090
umbral 0.65 -> F1 0.3786
umbral 0.70 -> F1 0.2681
umbral 0.75 -> F1 0.1284
umbral 0.80 -> F1 0.0151
umbral 0.85 -> F1 0.0000
umbral 0.90 -> F1 0.0000
umbral 0.95 -> F1 0.0000

Mejor umbral: 0.15

RESULTADOS EN TEST CON EL NUEVO UMBRAL
Accuracy : 0.8755
Precision: 0.882
Recall   : 0.867
F1-score : 0.8744

Matriz de confusión:
[[884 116]
 [133 867]]


In [12]:
import os
from google.colab import files

modelo.save("modelo_perros_bolivar_scratch.keras")

ruta = "modelo_perros_bolivar_scratch.keras"
print("Existe:", os.path.exists(ruta))
print("Tamaño:", round(os.path.getsize(ruta) / (1024 * 1024), 2), "MB")

files.download(ruta)

Existe: True
Tamaño: 5.03 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

def similitud_coseno(a, b):
    num = np.sum(a * b, axis=1)
    den = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
    return num / np.maximum(den, 1e-9)

# Huellas de validación (últimos 1600 pares de train) y de test
sim_val = similitud_coseno(
    extractor.predict(X1_val, verbose=0),
    extractor.predict(X2_val, verbose=0),
)
sim_test = similitud_coseno(
    extractor.predict(X1_test, verbose=0),
    extractor.predict(X2_test, verbose=0),
)

# Buscar el mejor umbral con validación
mejor_umbral, mejor_f1 = 0.5, 0
for umbral in np.arange(0.30, 1.00, 0.01):
    f1 = f1_score(y_val, (sim_val >= umbral).astype(int))
    if f1 > mejor_f1:
        mejor_umbral, mejor_f1 = umbral, f1

print(f"Mejor umbral de similitud: {mejor_umbral:.2f}  (F1 en validación: {mejor_f1:.4f})")

# Probarlo en perros nuevos
clase = (sim_test >= mejor_umbral).astype(int)

print("\nRESULTADOS EN TEST CON SIMILITUD COSENO")
print("Accuracy :", round(accuracy_score(y_test, clase), 4))
print("Precision:", round(precision_score(y_test, clase), 4))
print("Recall   :", round(recall_score(y_test, clase), 4))
print("F1-score :", round(f1_score(y_test, clase), 4))
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, clase))

Mejor umbral de similitud: 0.90  (F1 en validación: 0.9009)

RESULTADOS EN TEST CON SIMILITUD COSENO
Accuracy : 0.8755
Precision: 0.8924
Recall   : 0.854
F1-score : 0.8728

Matriz de confusión:
[[897 103]
 [146 854]]


In [1]:
import random
import numpy as np

random.seed(7)
perros_ranking = random.sample(todos_test, 100)   # perros que la red nunca vio

galeria_imgs, consultas_imgs = [], []
for nombre, imagenes in perros_ranking:
    foto_registrada, foto_busqueda = random.sample(imagenes, 2)
    galeria_imgs.append(cargar_imagen(foto_registrada))
    consultas_imgs.append(cargar_imagen(foto_busqueda))

galeria = extractor.predict(np.array(galeria_imgs), verbose=0)
consultas = extractor.predict(np.array(consultas_imgs), verbose=0)

def normalizar(m):
    return m / np.maximum(np.linalg.norm(m, axis=1, keepdims=True), 1e-9)

similitudes = normalizar(consultas) @ normalizar(galeria).T   # 100 x 100

top1 = top3 = top5 = 0
for i in range(len(similitudes)):
    orden = np.argsort(-similitudes[i])
    posicion = int(np.where(orden == i)[0][0])
    top1 += posicion < 1
    top3 += posicion < 3
    top5 += posicion < 5

print("Entre 100 perros registrados, buscando con otra foto del mismo perro:")
print(f"  Queda #1:           {top1} de 100")
print(f"  Queda entre los 3:  {top3} de 100")
print(f"  Queda entre los 5:  {top5} de 100")
print("(Al azar sería: 1, 3 y 5 de 100)")

NameError: name 'todos_test' is not defined

In [2]:
!wget -q -O "YT-BB-Dog.zip" "https://www.lirmm.fr/YT-BB-Dog_Sibetan/files/YT-BB-dog.zip"
!unzip -q "YT-BB-Dog.zip" -d "dataset"
!unzip -q "dataset/YT-BB-dog/YT-BB-Dog.zip" -d "dataset/perros"
!ls dataset/perros/YT-BB-Dog

test  train


In [3]:
import tensorflow as tf
from tensorflow.keras import layers
from google.colab import files

@tf.keras.utils.register_keras_serializable(package="Perros")
class AbsDifference(layers.Layer):
    def call(self, inputs):
        return tf.abs(inputs[0] - inputs[1])

subido = files.upload()
nombre_modelo = list(subido.keys())[0]

modelo = tf.keras.models.load_model(
    nombre_modelo,
    custom_objects={"AbsDifference": AbsDifference},
    safe_mode=False,
    compile=False,
)
extractor = modelo.get_layer("extractor_128")
print("Modelo cargado:", nombre_modelo)
print("Salida del extractor:", extractor.output_shape)

Saving modelo_perros_bolivar_scratch.keras to modelo_perros_bolivar_scratch.keras
Modelo cargado: modelo_perros_bolivar_scratch.keras
Salida del extractor: (None, 128)


In [4]:
import os
import random
from glob import glob
import cv2
import numpy as np

TAMANO = 128
base = "dataset/perros/YT-BB-Dog"

def cargar_imagen(ruta):
    imagen = cv2.imread(ruta)
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    imagen = cv2.resize(imagen, (TAMANO, TAMANO))
    return imagen.astype("float32") / 255.0

def obtener_perros(ruta):
    perros = []
    for nombre in os.listdir(ruta):
        carpeta = os.path.join(ruta, nombre)
        if os.path.isdir(carpeta):
            imagenes = glob(os.path.join(carpeta, "*.jpg"))
            if len(imagenes) >= 2:
                perros.append((nombre, imagenes))
    return perros

# Perros de la carpeta test: la red nunca los vio al entrenar
todos_test = obtener_perros(os.path.join(base, "test"))

random.seed(7)
perros_ranking = random.sample(todos_test, 100)

galeria_imgs, consultas_imgs = [], []
for nombre, imagenes in perros_ranking:
    foto_registrada, foto_busqueda = random.sample(imagenes, 2)
    galeria_imgs.append(cargar_imagen(foto_registrada))
    consultas_imgs.append(cargar_imagen(foto_busqueda))

galeria = extractor.predict(np.array(galeria_imgs), verbose=0)
consultas = extractor.predict(np.array(consultas_imgs), verbose=0)

def normalizar(m):
    return m / np.maximum(np.linalg.norm(m, axis=1, keepdims=True), 1e-9)

similitudes = normalizar(consultas) @ normalizar(galeria).T   # 100 x 100

top1 = top3 = top5 = 0
for i in range(len(similitudes)):
    orden = np.argsort(-similitudes[i])
    posicion = int(np.where(orden == i)[0][0])
    top1 += posicion < 1
    top3 += posicion < 3
    top5 += posicion < 5

print("Entre 100 perros registrados, buscando con otra foto del mismo perro:")
print(f"  Queda #1:           {top1} de 100")
print(f"  Queda entre los 3:  {top3} de 100")
print(f"  Queda entre los 5:  {top5} de 100")
print("(Al azar sería: 1, 3 y 5 de 100)")

Entre 100 perros registrados, buscando con otra foto del mismo perro:
  Queda #1:           54 de 100
  Queda entre los 3:  77 de 100
  Queda entre los 5:  78 de 100
(Al azar sería: 1, 3 y 5 de 100)
